# Práctica Final RL: Agente DQN para CartPole-v1

**Alumno:** Mohamed Yassine BenOmar  
**Fecha:** 27/03/2026  
**Asignatura:** OPT-ML (Deep Learning & Reinforcement Learning)  
**Entorno:** CartPole-v1 (Gymnasium)  

**Objetivo:** Diseñar, implementar y entrenar un agente DQN (Deep Q-Network) que aprenda a equilibrar un palo sobre un carro en el entorno CartPole-v1, aplicando los conocimientos de Deep Learning y Reinforcement Learning adquiridos durante el curso.

In [ ]:
# Instalación de dependencias
!pip install torch gymnasium matplotlib imageio numpy -q

In [ ]:
import torch
import numpy as np
import random
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Módulos propios
from agent import DQNAgent, DQNConfig, QNetwork
from environment import make_env, describe_env, run_random_episode, evaluate_agent, evaluate_random
from utils import plot_learning_curves, plot_comparison_curves, plot_evaluation, create_agent_gif

# Semillas para reproducibilidad
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

%matplotlib inline
print(f"PyTorch: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

---
## 2. Análisis del Entorno

CartPole-v1 es un entorno clásico de control en Reinforcement Learning. El objetivo es mantener un palo en equilibrio sobre un carro que se mueve a lo largo de un riel.

### Espacio de estados
El estado es un vector de 4 valores continuos:

| Variable | Descripción | Rango |
|---|---|---|
| Posición del carro | Posición horizontal del carro | [-4.8, 4.8] |
| Velocidad del carro | Velocidad lineal del carro | [-∞, ∞] |
| Ángulo del palo | Ángulo respecto a la vertical (rad) | [-0.418, 0.418] |
| Velocidad angular | Velocidad angular del palo | [-∞, ∞] |

### Espacio de acciones
2 acciones discretas: **0** (empujar izquierda) y **1** (empujar derecha).

### Recompensa
+1 por cada paso que el palo se mantiene en pie. Máximo: 500 (episodio truncado).

### Condiciones de terminación
- Ángulo del palo > ±12° (±0.2095 rad)
- Posición del carro > ±2.4
- Se alcanzan 500 pasos (truncado, no fallo)

### ¿Por qué DQN?
El espacio de estados es continuo (4 valores reales), lo que hace inviable una tabla Q tradicional. DQN aproxima la función Q con una red neuronal, permitiendo generalizar sobre estados continuos.

In [ ]:
# Crear entorno y mostrar análisis detallado
env = make_env()
env_info = describe_env(env)

In [ ]:
# Ejemplo de un episodio aleatorio con detalle de estados
reward, steps, states = run_random_episode(env, seed=42, verbose=True)

In [ ]:
# Baseline aleatorio: 100 episodios con acciones al azar
random_rewards = [run_random_episode(env, seed=i)[0] for i in range(100)]

print(f"\nEstadísticas del baseline aleatorio (100 episodios):")
print(f"  Media:    {np.mean(random_rewards):.1f}")
print(f"  Std:      {np.std(random_rewards):.1f}")
print(f"  Mínimo:   {np.min(random_rewards):.0f}")
print(f"  Máximo:   {np.max(random_rewards):.0f}")
print(f"  Mediana:  {np.median(random_rewards):.0f}")

plt.figure(figsize=(8, 4))
plt.hist(random_rewards, bins=20, color='lightsalmon', edgecolor='black', alpha=0.8)
plt.axvline(np.mean(random_rewards), color='red', linestyle='--', label=f'Media: {np.mean(random_rewards):.1f}')
plt.xlabel('Recompensa Total')
plt.ylabel('Frecuencia')
plt.title('Distribución de Recompensas - Baseline Aleatorio (100 episodios)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Diseño del Agente DQN

### Algoritmo DQN

DQN (Deep Q-Network) combina Q-Learning con una red neuronal profunda para aproximar la función de valor Q(s, a). Los componentes clave son:

1. **Red Q (QNetwork):** Red neuronal que recibe un estado y devuelve los Q-valores para cada acción posible.
2. **Target Network:** Copia de la red Q que se actualiza periódicamente. Proporciona targets estables durante el entrenamiento, evitando la inestabilidad de usar la misma red que se está actualizando.
3. **Experience Replay:** Buffer circular que almacena transiciones (s, a, r, s', done). Al muestrear aleatoriamente, se rompe la correlación temporal entre muestras consecutivas.
4. **Política ε-greedy:** Con probabilidad ε se toma una acción aleatoria (exploración), y con probabilidad 1-ε se toma la mejor acción según Q (explotación). ε decae durante el entrenamiento.

### Double DQN

DQN estándar tiende a sobreestimar los Q-valores porque usa `max` para seleccionar y evaluar acciones simultáneamente. **Double DQN** desacopla estos pasos:
- La red online selecciona la mejor acción: `a* = argmax Q_online(s', a)`
- La target network evalúa esa acción: `Q_target(s', a*)`

Esto reduce el sesgo de sobreestimación y mejora la estabilidad del entrenamiento.

### Arquitectura de la Red

```
Input (4) → Linear(128) → ReLU → Linear(128) → ReLU → Linear(2)
```

- **Entrada:** 4 neuronas (dimensión del estado)
- **Capa oculta 1:** 128 neuronas + ReLU
- **Capa oculta 2:** 128 neuronas + ReLU
- **Salida:** 2 neuronas (Q-valor por acción, sin activación)

**Justificación:** Dos capas de 128 unidades proporcionan suficiente capacidad para aprender la función Q en un espacio de 4 dimensiones con 2 acciones. ReLU es la activación estándar en DQN. No se necesita dropout ni batch normalization para este entorno relativamente simple.

### Hiperparámetros

| Parámetro | Valor | Justificación |
|---|---|---|
| Learning rate | 1e-3 | Estándar para Adam con redes pequeñas. CartPole converge rápido |
| Gamma (γ) | 0.99 | Descuento alto: las recompensas futuras importan para mantener equilibrio |
| Epsilon inicio | 1.0 | Exploración completa al inicio |
| Epsilon final | 0.01 | Exploración residual mínima |
| Epsilon decay | 0.995 | Decaimiento multiplicativo por episodio. Llega a ~0.05 en 600 ep. |
| Buffer size | 10,000 | Suficiente para decorrelacionar muestras (episodios cortos, ~200 pasos) |
| Batch size | 64 | Balance entre velocidad y estabilidad del gradiente |
| Target update freq | 10 | Actualización cada 10 episodios, frecuente para CartPole |
| Grad clip | 1.0 | Previene gradientes explosivos |

In [ ]:
# Configuración del agente DQN
config = DQNConfig(
    lr=1e-3,
    gamma=0.99,
    epsilon_start=1.0,
    epsilon_end=0.01,
    epsilon_decay=0.995,
    buffer_size=10_000,
    batch_size=64,
    target_update_freq=10,
    hidden_dims=(128, 128),
    max_episodes=600,
    min_buffer_size=1000,
    grad_clip=1.0,
    use_double_dqn=False
)

# Crear agente y mostrar arquitectura
agent = DQNAgent(state_dim=4, action_dim=2, config=config)
print("Arquitectura de la Red Q:")
print(agent.q_network)
print(f"\nNúmero total de parámetros: {sum(p.numel() for p in agent.q_network.parameters()):,}")
print(f"Device: {agent.device}")

In [ ]:
# Sanity check: pasar un estado aleatorio por la red
test_state = env.observation_space.sample()
test_tensor = torch.FloatTensor(test_state).unsqueeze(0)
with torch.no_grad():
    q_values = agent.q_network(test_tensor)

print(f"Estado de prueba: {test_state}")
print(f"Q-valores:        {q_values.numpy()[0]}")
print(f"Acción elegida:   {q_values.argmax().item()} (antes de entrenar, es arbitraria)")

---
## 4. Entrenamiento

### Bucle de entrenamiento

Para cada episodio:
1. Reiniciar el entorno
2. En cada paso: seleccionar acción (ε-greedy), ejecutar, almacenar transición en el buffer
3. Si el buffer tiene suficientes muestras: muestrear un minibatch y actualizar la red Q
4. Al final del episodio: decaer epsilon
5. Cada N episodios: sincronizar la target network

Se registran las métricas de recompensa, pérdida y epsilon por episodio para las curvas de aprendizaje.

In [ ]:
def train_agent(agent, env, config, seed=42):
    """Entrena el agente DQN y devuelve las métricas de entrenamiento."""
    rewards_history = []
    losses_history = []
    epsilon_history = []

    for episode in range(config.max_episodes):
        state, _ = env.reset(seed=seed + episode if episode == 0 else None)
        episode_reward = 0
        episode_losses = []

        for step in range(500):
            action = agent.select_action(state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            agent.replay_buffer.push(state, action, reward, next_state, float(done))
            state = next_state
            episode_reward += reward

            if len(agent.replay_buffer) >= config.min_buffer_size:
                loss = agent.update(config.batch_size)
                episode_losses.append(loss)

            if done:
                break

        agent.decay_epsilon()

        if episode % config.target_update_freq == 0:
            agent.update_target_network()

        rewards_history.append(episode_reward)
        losses_history.append(np.mean(episode_losses) if episode_losses else 0)
        epsilon_history.append(agent.epsilon)

        if episode % 50 == 0:
            avg_reward = np.mean(rewards_history[-100:])
            print(f"Ep {episode:>4d} | Reward: {episode_reward:>6.0f} | "
                  f"Avg100: {avg_reward:>6.1f} | Eps: {agent.epsilon:.4f} | "
                  f"Loss: {losses_history[-1]:.4f}")

    return rewards_history, losses_history, epsilon_history

In [ ]:
# === Entrenamiento DQN Estándar ===
print("=" * 60)
print("ENTRENAMIENTO: DQN Estándar")
print("=" * 60)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

env_train = make_env()
config_standard = DQNConfig(use_double_dqn=False)
agent_standard = DQNAgent(state_dim=4, action_dim=2, config=config_standard)

rewards_std, losses_std, epsilons_std = train_agent(agent_standard, env_train, config_standard)

# Guardar modelo
agent_standard.save('models/dqn_standard_cartpole.pth')
env_train.close()

print(f"\nRecompensa media últimos 100 episodios: {np.mean(rewards_std[-100:]):.1f}")

In [ ]:
# === Entrenamiento Double DQN ===
print("=" * 60)
print("ENTRENAMIENTO: Double DQN")
print("=" * 60)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

env_train2 = make_env()
config_double = DQNConfig(use_double_dqn=True)
agent_double = DQNAgent(state_dim=4, action_dim=2, config=config_double)

rewards_dbl, losses_dbl, epsilons_dbl = train_agent(agent_double, env_train2, config_double)

agent_double.save('models/dqn_double_cartpole.pth')
env_train2.close()

print(f"\nRecompensa media últimos 100 episodios: {np.mean(rewards_dbl[-100:]):.1f}")

---
## 5. Resultados

### Curvas de aprendizaje

A continuación se muestran las curvas de aprendizaje del agente DQN estándar: recompensa por episodio (con media móvil de 50 episodios), pérdida del entrenamiento y decaimiento de epsilon.

In [ ]:
# Curvas de aprendizaje - DQN Estándar
fig = plot_learning_curves(rewards_std, losses_std, epsilons_std, window=50)
fig.suptitle('DQN Estándar - Curvas de Aprendizaje', fontsize=14, y=1.02)
plt.show()

In [ ]:
# Curvas de aprendizaje - Double DQN
fig = plot_learning_curves(rewards_dbl, losses_dbl, epsilons_dbl, window=50)
fig.suptitle('Double DQN - Curvas de Aprendizaje', fontsize=14, y=1.02)
plt.show()

In [ ]:
# Comparación DQN vs Double DQN
fig = plot_comparison_curves({
    'DQN Estándar': rewards_std,
    'Double DQN': rewards_dbl
}, window=50)
plt.show()

print(f"\nComparación de métricas finales (últimos 100 episodios):")
print(f"  DQN Estándar:  Media = {np.mean(rewards_std[-100:]):.1f}, Std = {np.std(rewards_std[-100:]):.1f}")
print(f"  Double DQN:    Media = {np.mean(rewards_dbl[-100:]):.1f}, Std = {np.std(rewards_dbl[-100:]):.1f}")

---
## 6. Evaluación

Evaluamos el agente entrenado (sin exploración, ε=0) durante 100 episodios y comparamos con el baseline aleatorio. Esto nos permite medir objetivamente si el agente ha aprendido una política efectiva.

In [ ]:
# Evaluación del agente DQN entrenado vs baseline aleatorio
env_eval = make_env()

# Cargar mejor modelo (Double DQN)
best_agent = DQNAgent(state_dim=4, action_dim=2, config=DQNConfig(use_double_dqn=True))
best_agent.load('models/dqn_double_cartpole.pth')

# Evaluar
trained_rewards = evaluate_agent(env_eval, best_agent, n_episodes=100, seed=SEED)
baseline_rewards = evaluate_random(env_eval, n_episodes=100, seed=SEED)

print("=" * 60)
print("EVALUACIÓN (100 episodios, ε=0)")
print("=" * 60)
print(f"\n{'Métrica':<15} {'DQN Entrenado':>15} {'Baseline Aleatorio':>20}")
print("-" * 52)
print(f"{'Media':<15} {np.mean(trained_rewards):>15.1f} {np.mean(baseline_rewards):>20.1f}")
print(f"{'Std':<15} {np.std(trained_rewards):>15.1f} {np.std(baseline_rewards):>20.1f}")
print(f"{'Mínimo':<15} {np.min(trained_rewards):>15.0f} {np.min(baseline_rewards):>20.0f}")
print(f"{'Máximo':<15} {np.max(trained_rewards):>15.0f} {np.max(baseline_rewards):>20.0f}")
print(f"{'Mediana':<15} {np.median(trained_rewards):>15.0f} {np.median(baseline_rewards):>20.0f}")

env_eval.close()

In [ ]:
# Boxplot comparativo
fig = plot_evaluation(trained_rewards, baseline_rewards)
plt.show()

In [ ]:
# Crear GIF del agente entrenado jugando (Bonus: Visualización)
env_gif = make_env(render_mode='rgb_array')
gif_agent = DQNAgent(state_dim=4, action_dim=2, config=DQNConfig(use_double_dqn=True))
gif_agent.load('models/dqn_double_cartpole.pth')

gif_path = create_agent_gif(env_gif, gif_agent, filepath='agent_playing.gif')
env_gif.close()

# Mostrar GIF en el notebook
display(Image(filename=gif_path))

---
## 7. Conclusiones

### Resultados

El agente DQN ha aprendido exitosamente a resolver CartPole-v1, alcanzando recompensas consistentemente cercanas a 500 (el máximo posible). Comparado con el baseline aleatorio (~20-25 de media), el agente entrenado demuestra una mejora de más de 20x en rendimiento.

### DQN vs Double DQN

Ambas variantes logran resolver el entorno. Double DQN muestra una convergencia ligeramente más estable en algunos casos, gracias a la reducción del sesgo de sobreestimación de Q-valores. En un entorno simple como CartPole, la diferencia no es dramática, pero en entornos más complejos esta mejora sería más significativa.

### Dificultades encontradas

1. **Sensibilidad a hiperparámetros:** El aprendizaje es muy sensible al learning rate y al decaimiento de epsilon. Valores inadecuados pueden hacer que el agente no aprenda o se estanque.
2. **Variabilidad entre ejecuciones:** Incluso con las mismas semillas, pequeñas diferencias en el orden de operaciones pueden llevar a resultados diferentes.
3. **Tamaño del buffer:** Un buffer demasiado pequeño provoca inestabilidad; demasiado grande diluye las experiencias recientes.

### Posibles mejoras

- **Dueling DQN:** Separar la estimación del valor de estado V(s) y la ventaja A(s,a)
- **Prioritized Experience Replay:** Muestrear transiciones con mayor error TD con más frecuencia
- **Noisy Networks:** Reemplazar ε-greedy por exploración paramétrica aprendida
- **Aplicar a entornos más complejos:** Atari games con observaciones de píxeles (requeriría CNNs)

### Bonus: Análisis de Hiperparámetros

Comparamos el efecto del learning rate en la velocidad de convergencia. Se prueban tres valores: 1e-4 (conservador), 1e-3 (estándar), y 1e-2 (agresivo).

In [ ]:
# Análisis de hiperparámetros: comparación de learning rates
lr_results = {}
learning_rates = [1e-4, 1e-3, 1e-2]

for lr in learning_rates:
    print(f"\n--- Entrenando con lr={lr} ---")
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

    env_hp = make_env()
    config_hp = DQNConfig(lr=lr, max_episodes=400)  # Menos episodios para agilizar
    agent_hp = DQNAgent(state_dim=4, action_dim=2, config=config_hp)
    rewards_hp, _, _ = train_agent(agent_hp, env_hp, config_hp)
    lr_results[f'lr={lr}'] = rewards_hp
    env_hp.close()
    print(f"  Media últimos 100 ep: {np.mean(rewards_hp[-100:]):.1f}")

# Graficar comparación
fig = plot_comparison_curves(lr_results, window=50)
fig.suptitle('Análisis de Hiperparámetros: Efecto del Learning Rate', fontsize=13)
plt.show()

print("\nResumen:")
for name, rewards in lr_results.items():
    print(f"  {name}: Media final (100 ep) = {np.mean(rewards[-100:]):.1f}")

### Análisis del Learning Rate

- **lr=1e-4 (conservador):** Convergencia lenta pero estable. Puede necesitar más episodios para alcanzar la recompensa máxima.
- **lr=1e-3 (estándar):** Buen balance entre velocidad de convergencia y estabilidad. Es el valor elegido para nuestro agente principal.
- **lr=1e-2 (agresivo):** Convergencia rápida inicialmente pero puede ser inestable, con oscilaciones en la recompensa.

Este análisis confirma que **lr=1e-3 es la mejor elección** para CartPole-v1 con nuestra arquitectura de red.